In [ ]:
!uv pip install ultralytics
import ultralytics
ultralytics.checks()

## Loading the dataset

In [ ]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_KEY'))  # paste the token from discord
from datasets import load_dataset

ds = load_dataset("regularpooria/canopymap_plants")
print(ds)

## 1. Detection

YOLO26 _detection_ models have no suffix and are the default YOLO26 models, i.e. `yolo26n.pt` and are pretrained on COCO. See [Detection Docs](https://docs.ultralytics.com/tasks/detect) for full details.

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

# Load the pretrained YOLO26x model
model = YOLO("yolo26x.pt")

# Run prediction on the image
results = model(ds["train"][0]["image"])

# Display the image with bounding boxes
annotated_image = results[0].plot()

# Convert BGR (OpenCV) to RGB (Matplotlib)
annotated_image = annotated_image[:, :, ::-1]

plt.figure(figsize=(10, 8))
plt.imshow(annotated_image)
plt.axis("off")
plt.title("YOLO26 Object Detection")
plt.show()

In [ ]:
import gc

data = ds["train"]
BATCH_SIZE = 16
all_results = []

for i in range(0, len(data), BATCH_SIZE):
    batch = data[i:i+BATCH_SIZE]["image"]
    results = model.predict(batch, verbose=False, stream=True)  # generator, lower peak memory

    for r in results:
        all_results.append({
            "boxes_xyxy": r.boxes.xyxy.cpu().numpy(),
            "conf": r.boxes.conf.cpu().numpy(),
            "cls": r.boxes.cls.cpu().numpy().astype(int),
            # add r.masks.xy, r.keypoints.xy, etc. only if your model/task needs them
        })

    del batch, results
    gc.collect()

    print(f"Batch {i//BATCH_SIZE + 1}/{-(-len(data)//BATCH_SIZE)}")

print(len(all_results))  # 3163

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# Count detections per class across all images
counts = Counter()
for r in all_results:
    counts.update(r["cls"].tolist())

# Map class ids to names
labels_counts = [(model.names[c], n) for c, n in counts.items()]

def plot_counts(items, title):
    names = [x[0] for x in items]
    values = [x[1] for x in items]

    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(items))))
    bars = ax.barh(names, values, color="steelblue")
    ax.invert_yaxis()  # first item at the top
    ax.bar_label(bars, padding=3)
    ax.set_xlabel("Number of detections")
    ax.set_title(title)
    ax.margins(x=0.1)
    plt.tight_layout()
    plt.show()

# High to low
plot_counts(sorted(labels_counts, key=lambda x: x[1], reverse=True),
            "Detections per class (high to low)")

# Low to high
plot_counts(sorted(labels_counts, key=lambda x: x[1]),
            "Detections per class (low to high)")

In [ ]:
BLACKLIST = [
    #"scissors",
    #"person"
    "clock"
    # ""
    # add more class names here
]

# Drop empty strings and check the names exist in the model
BLACKLIST = [b for b in BLACKLIST if b.strip()]
valid_names = set(model.names.values())
unknown = [b for b in BLACKLIST if b not in valid_names]
if unknown:
    print("Not valid class names:", unknown)

blacklist_ids = {cid for cid, name in model.names.items() if name in BLACKLIST}
print("Blacklisted class ids:", blacklist_ids)

In [ ]:
MIN_CONF = 0.3  # raise this (e.g. 0.5) to ignore weak detections

flagged_idx = []
for i, r in enumerate(all_results):
    hit = (
        set(r["cls"].tolist()) & blacklist_ids
        if MIN_CONF == 0.0
        else {c for c, p in zip(r["cls"].tolist(), r["conf"].tolist())
              if c in blacklist_ids and p >= MIN_CONF}
    )
    if hit:
        flagged_idx.append(i)

print(f"{len(flagged_idx)} / {len(all_results)} images contain a blacklisted class")

In [ ]:
flagged_ds = ds["train"].select(flagged_idx)
print(flagged_ds)

In [ ]:
import matplotlib.pyplot as plt

MAX_SHOW = 17  # set to None to show every flagged image

to_show = flagged_idx if MAX_SHOW is None else flagged_idx[:MAX_SHOW]

for k, idx in enumerate(to_show):
    image = flagged_ds[k]["image"].convert("RGB")

    # Re-run prediction on this single flagged image
    result = model.predict(image, verbose=False)[0]

    # Display the image with bounding boxes
    annotated_image = result.plot()

    # Convert BGR (OpenCV) to RGB (Matplotlib)
    annotated_image = annotated_image[:, :, ::-1]

    detected = sorted({model.names[int(c)] for c in result.boxes.cls})

    plt.figure(figsize=(10, 8))
    plt.imshow(annotated_image)
    plt.axis("off")
    plt.title(f"Dataset idx {idx} | Detected: {', '.join(detected)}")
    plt.show()